# Escalabilidad y Sostenibilidad en Sistemas de IA

## Objetivo
En este notebook implementaremos patrones de escalabilidad para sistemas de IA en producción:
caché inteligente, procesamiento por lotes, enrutamiento de modelos, estimación de costos
y patrones de resiliencia.

## ¿Por qué importa la escalabilidad?

Cuando un sistema de IA pasa de prototipo a producción, enfrenta desafíos críticos:

### Costos
- Las APIs de LLMs cobran por token. Sin optimización, los costos pueden escalar linealmente con el número de usuarios.
- Un solo modelo GPT-4o procesando 10,000 consultas/día puede costar **$250-500 USD/día**.
- Con caché y enrutamiento inteligente, estos costos pueden reducirse en un **60-80%**.

### Latencia
- Los usuarios esperan respuestas en menos de 2 segundos.
- Una llamada a GPT-4o puede tomar 3-10 segundos.
- El caché puede reducir la latencia a **milisegundos** para consultas repetidas.

### Disponibilidad
- Las APIs externas pueden fallar o tener interrupciones.
- Sin patrones de resiliencia, una caída de la API significa una caída del sistema completo.

### Sostenibilidad
- El consumo energético de los LLMs es significativo.
- Reducir llamadas innecesarias no solo ahorra dinero, sino que reduce la huella de carbono.

---

En este notebook construiremos soluciones prácticas para cada uno de estos desafíos.

## 0. Configuración del Entorno

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

for env_path in [Path.cwd() / ".env", *[parent / ".env" for parent in Path.cwd().parents]]:
    if env_path.exists():
        load_dotenv(env_path)
        break

if not os.getenv("GITHUB_TOKEN"):
    raise EnvironmentError("GITHUB_TOKEN no configurado. Copia .env.example a .env y agrega tu token.")
print("Entorno configurado correctamente")

In [ ]:
from openai import OpenAI
import json
import time
import hashlib
import re
from datetime import datetime, timedelta
from collections import defaultdict
from dataclasses import dataclass, field

client = OpenAI(
    base_url=os.getenv("GITHUB_BASE_URL", "https://models.inference.ai.azure.com"),
    api_key=os.getenv("GITHUB_TOKEN")
)

MODELO_LIGERO = "gpt-4o-mini"
MODELO_POTENTE = "gpt-4o"

print(f"Cliente configurado. Modelos: {MODELO_LIGERO} (ligero), {MODELO_POTENTE} (potente)")

## 1. Caché Inteligente para LLMs

Un caché inteligente va más allá de la coincidencia exacta: usa similitud semántica
para reutilizar respuestas de preguntas similares. También incluye TTL (tiempo de vida)
para mantener las respuestas actualizadas.

In [ ]:
import math


@dataclass
class EntradaCache:
    """Una entrada en el caché semántico."""
    consulta: str
    respuesta: str
    timestamp: float
    ttl_segundos: float
    tokens_ahorrados: int = 0
    accesos: int = 0

    def esta_vigente(self) -> bool:
        """Verifica si la entrada aún no ha expirado."""
        return (time.time() - self.timestamp) < self.ttl_segundos


class CacheSemantico:
    """Caché que usa similitud semántica para encontrar respuestas previas."""

    def __init__(self, umbral_similitud: float = 0.85, ttl_segundos: float = 3600):
        """
        umbral_similitud: Similitud mínima para considerar un hit de caché (0.0 a 1.0).
        ttl_segundos: Tiempo de vida de las entradas en segundos.
        """
        self.umbral_similitud = umbral_similitud
        self.ttl_segundos = ttl_segundos
        self.entradas: list[EntradaCache] = []
        self.estadisticas = {
            "hits": 0,
            "misses": 0,
            "tokens_ahorrados_total": 0,
            "costo_ahorrado_usd": 0.0
        }

    def _normalizar_texto(self, texto: str) -> str:
        """Normaliza el texto para mejorar la comparación."""
        texto = texto.lower().strip()
        texto = re.sub(r'[^\w\s]', '', texto)  # Quitar puntuación
        texto = re.sub(r'\s+', ' ', texto)       # Normalizar espacios
        return texto

    def _similitud_coseno(self, texto1: str, texto2: str) -> float:
        """Calcula similitud coseno basada en frecuencia de palabras."""
        def obtener_freq(texto):
            palabras = texto.split()
            freq = defaultdict(int)
            for p in palabras:
                freq[p] += 1
            return freq

        t1 = self._normalizar_texto(texto1)
        t2 = self._normalizar_texto(texto2)

        freq1 = obtener_freq(t1)
        freq2 = obtener_freq(t2)

        todas = set(freq1.keys()) | set(freq2.keys())
        producto = sum(freq1.get(p, 0) * freq2.get(p, 0) for p in todas)
        mag1 = math.sqrt(sum(v**2 for v in freq1.values()))
        mag2 = math.sqrt(sum(v**2 for v in freq2.values()))

        if mag1 == 0 or mag2 == 0:
            return 0.0
        return producto / (mag1 * mag2)

    def buscar(self, consulta: str) -> tuple[bool, str | None, float]:
        """Busca en el caché una respuesta similar a la consulta.

        Retorna: (encontrado, respuesta, similitud)
        """
        mejor_similitud = 0.0
        mejor_entrada = None

        for entrada in self.entradas:
            if not entrada.esta_vigente():
                continue

            similitud = self._similitud_coseno(consulta, entrada.consulta)
            if similitud > mejor_similitud:
                mejor_similitud = similitud
                mejor_entrada = entrada

        if mejor_entrada and mejor_similitud >= self.umbral_similitud:
            mejor_entrada.accesos += 1
            self.estadisticas["hits"] += 1
            return True, mejor_entrada.respuesta, mejor_similitud
        else:
            self.estadisticas["misses"] += 1
            return False, None, mejor_similitud

    def guardar(self, consulta: str, respuesta: str, tokens_usados: int = 0):
        """Guarda una nueva entrada en el caché."""
        entrada = EntradaCache(
            consulta=consulta,
            respuesta=respuesta,
            timestamp=time.time(),
            ttl_segundos=self.ttl_segundos,
            tokens_ahorrados=tokens_usados
        )
        self.entradas.append(entrada)

    def registrar_ahorro(self, tokens: int, costo_por_1k: float = 0.00075):
        """Registra tokens y costo ahorrados por un cache hit."""
        self.estadisticas["tokens_ahorrados_total"] += tokens
        self.estadisticas["costo_ahorrado_usd"] += (tokens / 1000) * costo_por_1k

    def limpiar_expirados(self):
        """Elimina entradas expiradas del caché."""
        antes = len(self.entradas)
        self.entradas = [e for e in self.entradas if e.esta_vigente()]
        return antes - len(self.entradas)

    def obtener_estadisticas(self) -> dict:
        """Retorna estadísticas del caché."""
        total = self.estadisticas["hits"] + self.estadisticas["misses"]
        tasa_hits = (self.estadisticas["hits"] / total * 100) if total > 0 else 0
        return {
            **self.estadisticas,
            "total_consultas": total,
            "tasa_hits": round(tasa_hits, 2),
            "entradas_activas": len([e for e in self.entradas if e.esta_vigente()]),
            "entradas_total": len(self.entradas)
        }


print("CacheSemantico creado correctamente")

In [ ]:
# Probar el caché semántico
cache = CacheSemantico(umbral_similitud=0.75, ttl_segundos=3600)

# Función auxiliar para consultar con caché
def consultar_con_cache(pregunta: str, cache: CacheSemantico) -> dict:
    """Realiza una consulta usando caché semántico."""
    inicio = time.time()

    # Buscar en caché
    encontrado, respuesta_cache, similitud = cache.buscar(pregunta)

    if encontrado:
        latencia = (time.time() - inicio) * 1000  # ms
        cache.registrar_ahorro(tokens=150)  # Estimación
        return {
            "fuente": "CACHE",
            "respuesta": respuesta_cache,
            "similitud": round(similitud, 4),
            "latencia_ms": round(latencia, 2),
            "tokens_usados": 0
        }

    # Si no está en caché, consultar al modelo
    resultado = client.chat.completions.create(
        model=MODELO_LIGERO,
        messages=[
            {"role": "system", "content": "Responde de forma concisa."},
            {"role": "user", "content": pregunta}
        ],
        temperature=0.3,
        max_tokens=200
    )

    respuesta = resultado.choices[0].message.content
    tokens = resultado.usage.total_tokens
    latencia = (time.time() - inicio) * 1000

    # Guardar en caché
    cache.guardar(pregunta, respuesta, tokens)

    return {
        "fuente": "API",
        "respuesta": respuesta,
        "similitud": 0.0,
        "latencia_ms": round(latencia, 2),
        "tokens_usados": tokens
    }


# Simular consultas (algunas repetidas o similares)
consultas = [
    "¿Qué es machine learning?",
    "¿Qué es inteligencia artificial?",
    "¿Qué es el machine learning?",       # Similar a la primera
    "Explícame qué es machine learning",    # Similar a la primera
    "¿Qué es deep learning?",
    "¿Qué es la inteligencia artificial?",  # Similar a la segunda
    "¿Qué es Python?",
    "¿Qué es el deep learning?",            # Similar a la quinta
]

print("Procesando consultas con caché semántico...\n")
for consulta in consultas:
    resultado = consultar_con_cache(consulta, cache)
    icono = "⚡" if resultado["fuente"] == "CACHE" else "🌐"
    print(f"{icono} [{resultado['fuente']:5}] {consulta}")
    print(f"   Latencia: {resultado['latencia_ms']:.1f}ms | "
          f"Tokens: {resultado['tokens_usados']} | "
          f"Similitud: {resultado['similitud']}")
    print(f"   Respuesta: {resultado['respuesta'][:100]}...\n")

# Mostrar estadísticas
stats = cache.obtener_estadisticas()
print(f"\n{'='*60}")
print(f"ESTADÍSTICAS DEL CACHÉ")
print(f"{'='*60}")
print(f"Total consultas: {stats['total_consultas']}")
print(f"Cache hits: {stats['hits']} ({stats['tasa_hits']}%)")
print(f"Cache misses: {stats['misses']}")
print(f"Tokens ahorrados: {stats['tokens_ahorrados_total']}")
print(f"Costo ahorrado: ${stats['costo_ahorrado_usd']:.4f} USD")
print(f"Entradas activas: {stats['entradas_activas']}")

## 2. Procesamiento por Lotes (Batch)

En lugar de procesar cada consulta individualmente, el procesamiento por lotes agrupa
solicitudes similares y las procesa juntas. Esto reduce la sobrecarga de red y permite
priorizar solicitudes críticas.

In [ ]:
import heapq
from enum import IntEnum


class Prioridad(IntEnum):
    """Niveles de prioridad para las solicitudes."""
    CRITICA = 1
    ALTA = 2
    NORMAL = 3
    BAJA = 4


@dataclass
class Solicitud:
    """Una solicitud en la cola de procesamiento."""
    id: str
    pregunta: str
    prioridad: Prioridad
    timestamp: float = field(default_factory=time.time)
    respuesta: str | None = None
    estado: str = "pendiente"  # pendiente, procesando, completada, error
    tokens_usados: int = 0
    latencia_ms: float = 0.0

    def __lt__(self, other):
        """Comparación para la cola de prioridad."""
        if self.prioridad != other.prioridad:
            return self.prioridad < other.prioridad
        return self.timestamp < other.timestamp


class ProcesadorLotes:
    """Procesa solicitudes en lotes con cola de prioridad."""

    def __init__(self, client: OpenAI, modelo: str = "gpt-4o-mini",
                 tamano_lote: int = 5):
        self.client = client
        self.modelo = modelo
        self.tamano_lote = tamano_lote
        self.cola: list[Solicitud] = []
        self.procesadas: list[Solicitud] = []
        self.contador_id = 0

    def agregar(self, pregunta: str, prioridad: Prioridad = Prioridad.NORMAL) -> str:
        """Agrega una solicitud a la cola de procesamiento."""
        self.contador_id += 1
        solicitud = Solicitud(
            id=f"SOL-{self.contador_id:04d}",
            pregunta=pregunta,
            prioridad=prioridad
        )
        heapq.heappush(self.cola, solicitud)
        return solicitud.id

    def _procesar_solicitud(self, solicitud: Solicitud) -> Solicitud:
        """Procesa una solicitud individual."""
        solicitud.estado = "procesando"
        inicio = time.time()

        try:
            resultado = self.client.chat.completions.create(
                model=self.modelo,
                messages=[
                    {"role": "system", "content": "Responde de forma concisa en 1-2 oraciones."},
                    {"role": "user", "content": solicitud.pregunta}
                ],
                temperature=0.3,
                max_tokens=150
            )
            solicitud.respuesta = resultado.choices[0].message.content
            solicitud.tokens_usados = resultado.usage.total_tokens
            solicitud.estado = "completada"
        except Exception as e:
            solicitud.respuesta = f"Error: {str(e)}"
            solicitud.estado = "error"

        solicitud.latencia_ms = (time.time() - inicio) * 1000
        return solicitud

    def procesar_lote(self) -> list[Solicitud]:
        """Procesa un lote de solicitudes según prioridad."""
        lote = []
        for _ in range(min(self.tamano_lote, len(self.cola))):
            if self.cola:
                lote.append(heapq.heappop(self.cola))

        if not lote:
            return []

        print(f"\nProcesando lote de {len(lote)} solicitudes...")
        resultados = []
        for i, solicitud in enumerate(lote):
            print(f"  [{i+1}/{len(lote)}] {solicitud.id} (P{solicitud.prioridad.value}: {solicitud.prioridad.name})")
            resultado = self._procesar_solicitud(solicitud)
            resultados.append(resultado)
            self.procesadas.append(resultado)

        return resultados

    def procesar_todo(self) -> list[Solicitud]:
        """Procesa todas las solicitudes en la cola por lotes."""
        todos_resultados = []
        lote_num = 0
        while self.cola:
            lote_num += 1
            print(f"\n{'='*40} Lote {lote_num} {'='*40}")
            resultados = self.procesar_lote()
            todos_resultados.extend(resultados)
        return todos_resultados

    def obtener_resumen(self) -> dict:
        """Genera un resumen del procesamiento."""
        if not self.procesadas:
            return {"mensaje": "No hay solicitudes procesadas"}

        completadas = [s for s in self.procesadas if s.estado == "completada"]
        errores = [s for s in self.procesadas if s.estado == "error"]
        tokens_total = sum(s.tokens_usados for s in self.procesadas)
        latencia_promedio = sum(s.latencia_ms for s in self.procesadas) / len(self.procesadas)

        return {
            "total_procesadas": len(self.procesadas),
            "completadas": len(completadas),
            "errores": len(errores),
            "pendientes": len(self.cola),
            "tokens_total": tokens_total,
            "latencia_promedio_ms": round(latencia_promedio, 2),
            "costo_estimado_usd": round((tokens_total / 1000) * 0.00075, 6)
        }


print("ProcesadorLotes creado correctamente")

In [ ]:
# Probar el procesamiento por lotes con prioridades
procesador = ProcesadorLotes(client, MODELO_LIGERO, tamano_lote=3)

# Agregar solicitudes con diferentes prioridades
solicitudes_prueba = [
    ("¿Qué es Python?", Prioridad.BAJA),
    ("¿Cuál es la diferencia entre lista y tupla en Python?", Prioridad.NORMAL),
    ("URGENTE: ¿Cómo manejo un error crítico en producción?", Prioridad.CRITICA),
    ("¿Qué es un decorador en Python?", Prioridad.NORMAL),
    ("¿Cómo optimizar una consulta SQL lenta?", Prioridad.ALTA),
    ("¿Qué es Docker?", Prioridad.BAJA),
    ("URGENTE: Servidor caído, ¿qué verifico primero?", Prioridad.CRITICA),
]

print("Agregando solicitudes a la cola...")
for pregunta, prioridad in solicitudes_prueba:
    id_sol = procesador.agregar(pregunta, prioridad)
    print(f"  {id_sol}: [{prioridad.name}] {pregunta[:50]}")

# Procesar todo
resultados = procesador.procesar_todo()

# Mostrar resultados
print(f"\n\n{'='*60}")
print("RESULTADOS")
print(f"{'='*60}")
for sol in resultados:
    print(f"\n{sol.id} [{sol.prioridad.name}] - {sol.estado.upper()}")
    print(f"  Pregunta: {sol.pregunta[:60]}")
    print(f"  Respuesta: {sol.respuesta[:120]}..." if sol.respuesta and len(sol.respuesta) > 120 else f"  Respuesta: {sol.respuesta}")
    print(f"  Tokens: {sol.tokens_usados} | Latencia: {sol.latencia_ms:.0f}ms")

# Resumen
resumen = procesador.obtener_resumen()
print(f"\n{'='*60}")
print("RESUMEN DEL PROCESAMIENTO")
for k, v in resumen.items():
    print(f"  {k}: {v}")

## 3. Enrutamiento de Modelos

No todas las consultas necesitan el modelo más potente (y caro).
El enrutamiento inteligente clasifica la complejidad de cada consulta y la dirige
al modelo más apropiado, optimizando el balance costo/calidad.

In [ ]:
class EnrutadorModelos:
    """Enruta consultas al modelo más apropiado según su complejidad."""

    # Costos por 1K tokens (input + output promedio)
    COSTOS = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006, "nombre": "GPT-4o Mini"},
        "gpt-4o": {"input": 0.0025, "output": 0.01, "nombre": "GPT-4o"},
    }

    def __init__(self, client: OpenAI):
        self.client = client
        self.historial = []

    def clasificar_complejidad(self, consulta: str) -> str:
        """Clasifica la complejidad de una consulta usando heurísticas."""
        consulta_lower = consulta.lower()

        # Indicadores de alta complejidad
        indicadores_complejos = [
            r"\banaliza\b", r"\bcompara\b", r"\bdiseña\b", r"\bescribe un (plan|ensayo|artículo)",
            r"\bpro y contra\b", r"\bventajas y desventajas\b",
            r"\bimplementa\b", r"\barchitectura\b", r"\bestrategia\b",
            r"\bcódigo completo\b", r"\bexplica en detalle\b",
            r"\bmúltiples (pasos|factores|perspectivas)\b",
        ]

        # Indicadores de complejidad media
        indicadores_medios = [
            r"\bexplica\b", r"\bcómo funciona\b", r"\bdiferencia entre\b",
            r"\bejemplo de\b", r"\bpor qué\b", r"\bresume\b",
        ]

        # Contar indicadores
        score_complejo = sum(1 for p in indicadores_complejos if re.search(p, consulta_lower))
        score_medio = sum(1 for p in indicadores_medios if re.search(p, consulta_lower))

        # Longitud como factor adicional
        palabras = len(consulta.split())

        if score_complejo >= 2 or (score_complejo >= 1 and palabras > 30):
            return "compleja"
        elif score_medio >= 1 or score_complejo >= 1 or palabras > 20:
            return "media"
        else:
            return "simple"

    def seleccionar_modelo(self, complejidad: str) -> str:
        """Selecciona el modelo según la complejidad."""
        if complejidad == "compleja":
            return "gpt-4o"
        else:  # simple o media
            return "gpt-4o-mini"

    def procesar(self, consulta: str) -> dict:
        """Procesa una consulta enrutada al modelo apropiado."""
        complejidad = self.clasificar_complejidad(consulta)
        modelo = self.seleccionar_modelo(complejidad)
        costos_modelo = self.COSTOS[modelo]

        inicio = time.time()
        try:
            resultado = self.client.chat.completions.create(
                model=modelo,
                messages=[
                    {"role": "system", "content": "Responde de forma clara y útil."},
                    {"role": "user", "content": consulta}
                ],
                temperature=0.5,
                max_tokens=300 if complejidad != "compleja" else 600
            )

            respuesta = resultado.choices[0].message.content
            uso = resultado.usage
            costo = (
                (uso.prompt_tokens / 1000) * costos_modelo["input"] +
                (uso.completion_tokens / 1000) * costos_modelo["output"]
            )

            # Calcular costo hipotético si siempre se usara gpt-4o
            costo_gpt4o = (
                (uso.prompt_tokens / 1000) * self.COSTOS["gpt-4o"]["input"] +
                (uso.completion_tokens / 1000) * self.COSTOS["gpt-4o"]["output"]
            )

            registro = {
                "consulta": consulta[:80],
                "complejidad": complejidad,
                "modelo_usado": modelo,
                "modelo_nombre": costos_modelo["nombre"],
                "respuesta": respuesta,
                "tokens": {"input": uso.prompt_tokens, "output": uso.completion_tokens, "total": uso.total_tokens},
                "costo_real": round(costo, 6),
                "costo_gpt4o": round(costo_gpt4o, 6),
                "ahorro": round(costo_gpt4o - costo, 6),
                "latencia_ms": round((time.time() - inicio) * 1000, 2)
            }
            self.historial.append(registro)
            return registro

        except Exception as e:
            return {"error": str(e), "modelo_usado": modelo, "complejidad": complejidad}

    def reporte_costos(self) -> dict:
        """Genera un reporte de costos comparativo."""
        if not self.historial:
            return {"mensaje": "Sin datos"}

        costo_real_total = sum(r["costo_real"] for r in self.historial)
        costo_gpt4o_total = sum(r["costo_gpt4o"] for r in self.historial)
        ahorro_total = costo_gpt4o_total - costo_real_total
        porcentaje_ahorro = (ahorro_total / costo_gpt4o_total * 100) if costo_gpt4o_total > 0 else 0

        por_complejidad = defaultdict(lambda: {"count": 0, "costo": 0.0})
        for r in self.historial:
            c = r["complejidad"]
            por_complejidad[c]["count"] += 1
            por_complejidad[c]["costo"] += r["costo_real"]

        return {
            "total_consultas": len(self.historial),
            "costo_real_total": round(costo_real_total, 6),
            "costo_todo_gpt4o": round(costo_gpt4o_total, 6),
            "ahorro_total": round(ahorro_total, 6),
            "porcentaje_ahorro": round(porcentaje_ahorro, 2),
            "por_complejidad": dict(por_complejidad)
        }


print("EnrutadorModelos creado correctamente")

In [ ]:
# Probar el enrutamiento de modelos
enrutador = EnrutadorModelos(client)

consultas_variadas = [
    "¿Qué hora es?",                                                    # Simple
    "¿Qué es Python?",                                                   # Simple
    "Explica cómo funciona un algoritmo de búsqueda binaria",             # Media
    "¿Cuál es la diferencia entre SQL y NoSQL?",                          # Media
    "Analiza las ventajas y desventajas de microservicios vs monolito y diseña una estrategia de migración",  # Compleja
    "¿Qué es REST?",                                                     # Simple
    "Compara los principales frameworks de machine learning y explica en detalle cuándo usar cada uno",  # Compleja
]

print("Procesando consultas con enrutamiento inteligente...\n")
for consulta in consultas_variadas:
    resultado = enrutador.procesar(consulta)
    if "error" not in resultado:
        print(f"[{resultado['complejidad'].upper():8}] -> {resultado['modelo_nombre']}")
        print(f"  Consulta: {consulta[:70]}")
        print(f"  Costo real: ${resultado['costo_real']:.6f} | "
              f"Si fuera GPT-4o: ${resultado['costo_gpt4o']:.6f} | "
              f"Ahorro: ${resultado['ahorro']:.6f}")
        print(f"  Latencia: {resultado['latencia_ms']:.0f}ms\n")
    else:
        print(f"  Error: {resultado['error']}\n")

# Reporte de costos
reporte = enrutador.reporte_costos()
print(f"\n{'='*60}")
print("REPORTE DE COSTOS - ENRUTAMIENTO")
print(f"{'='*60}")
print(f"Costo con enrutamiento: ${reporte['costo_real_total']:.6f} USD")
print(f"Costo sin enrutamiento (todo GPT-4o): ${reporte['costo_todo_gpt4o']:.6f} USD")
print(f"Ahorro total: ${reporte['ahorro_total']:.6f} USD ({reporte['porcentaje_ahorro']}%)")
print(f"\nDistribución por complejidad:")
for comp, datos in reporte["por_complejidad"].items():
    print(f"  {comp}: {datos['count']} consultas, ${datos['costo']:.6f} USD")

## 4. Estimación de Costos

Un calculador de costos permite proyectar gastos y configurar alertas de presupuesto.
Esto es fundamental para la sostenibilidad financiera del sistema.

In [ ]:
class CalculadorCostos:
    """Calcula, proyecta y monitorea costos de uso de LLMs."""

    PRECIOS_POR_1K_TOKENS = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.0025, "output": 0.01},
        "gpt-4-turbo": {"input": 0.01, "output": 0.03},
        "gpt-3.5-turbo": {"input": 0.0005, "output": 0.0015},
    }

    def __init__(self, presupuesto_diario: float = 10.0):
        self.presupuesto_diario = presupuesto_diario
        self.registro_uso: list[dict] = []
        self.alertas: list[str] = []

    def registrar(self, modelo: str, tokens_input: int, tokens_output: int):
        """Registra una transacción."""
        precios = self.PRECIOS_POR_1K_TOKENS.get(modelo, self.PRECIOS_POR_1K_TOKENS["gpt-4o-mini"])
        costo = (
            (tokens_input / 1000) * precios["input"] +
            (tokens_output / 1000) * precios["output"]
        )

        self.registro_uso.append({
            "timestamp": datetime.now().isoformat(),
            "modelo": modelo,
            "tokens_input": tokens_input,
            "tokens_output": tokens_output,
            "tokens_total": tokens_input + tokens_output,
            "costo_usd": round(costo, 8)
        })

        # Verificar alertas
        costo_total_hoy = self.costo_total()
        porcentaje = (costo_total_hoy / self.presupuesto_diario) * 100

        if porcentaje >= 100:
            self.alertas.append(f"CRITICO: Presupuesto diario excedido ({porcentaje:.1f}%)")
        elif porcentaje >= 80:
            self.alertas.append(f"ADVERTENCIA: {porcentaje:.1f}% del presupuesto diario usado")
        elif porcentaje >= 50:
            self.alertas.append(f"INFO: {porcentaje:.1f}% del presupuesto diario usado")

        return costo

    def costo_total(self) -> float:
        """Retorna el costo total acumulado."""
        return sum(r["costo_usd"] for r in self.registro_uso)

    def proyectar_costos(self, consultas_por_dia: int,
                         tokens_promedio_input: int = 150,
                         tokens_promedio_output: int = 200,
                         modelo: str = "gpt-4o-mini") -> dict:
        """Proyecta costos para un periodo dado."""
        precios = self.PRECIOS_POR_1K_TOKENS.get(modelo, self.PRECIOS_POR_1K_TOKENS["gpt-4o-mini"])
        costo_por_consulta = (
            (tokens_promedio_input / 1000) * precios["input"] +
            (tokens_promedio_output / 1000) * precios["output"]
        )

        costo_diario = costo_por_consulta * consultas_por_dia
        costo_mensual = costo_diario * 30
        costo_anual = costo_diario * 365

        return {
            "modelo": modelo,
            "consultas_por_dia": consultas_por_dia,
            "costo_por_consulta": round(costo_por_consulta, 6),
            "costo_diario": round(costo_diario, 4),
            "costo_mensual": round(costo_mensual, 2),
            "costo_anual": round(costo_anual, 2),
            "tokens_diarios": consultas_por_dia * (tokens_promedio_input + tokens_promedio_output)
        }

    def comparar_modelos(self, consultas_por_dia: int = 1000,
                         tokens_promedio_input: int = 150,
                         tokens_promedio_output: int = 200) -> list[dict]:
        """Compara costos entre todos los modelos disponibles."""
        comparaciones = []
        for modelo in self.PRECIOS_POR_1K_TOKENS:
            proyeccion = self.proyectar_costos(
                consultas_por_dia, tokens_promedio_input, tokens_promedio_output, modelo
            )
            comparaciones.append(proyeccion)

        # Ordenar por costo mensual
        comparaciones.sort(key=lambda x: x["costo_mensual"])
        return comparaciones

    def resumen(self) -> dict:
        """Genera resumen de uso y costos."""
        if not self.registro_uso:
            return {"mensaje": "Sin registros"}

        tokens_total = sum(r["tokens_total"] for r in self.registro_uso)
        costo_total = self.costo_total()

        por_modelo = defaultdict(lambda: {"consultas": 0, "tokens": 0, "costo": 0.0})
        for r in self.registro_uso:
            m = r["modelo"]
            por_modelo[m]["consultas"] += 1
            por_modelo[m]["tokens"] += r["tokens_total"]
            por_modelo[m]["costo"] += r["costo_usd"]

        return {
            "total_consultas": len(self.registro_uso),
            "tokens_total": tokens_total,
            "costo_total_usd": round(costo_total, 6),
            "presupuesto_restante": round(self.presupuesto_diario - costo_total, 6),
            "porcentaje_presupuesto": round((costo_total / self.presupuesto_diario) * 100, 2),
            "por_modelo": {k: {kk: round(vv, 6) if isinstance(vv, float) else vv for kk, vv in v.items()} for k, v in por_modelo.items()},
            "alertas": self.alertas[-5:]  # Ultimas 5 alertas
        }


print("CalculadorCostos creado correctamente")

In [ ]:
# Probar el calculador de costos
calculador = CalculadorCostos(presupuesto_diario=5.0)

# Simular uso
usos_simulados = [
    ("gpt-4o-mini", 100, 150),
    ("gpt-4o-mini", 200, 300),
    ("gpt-4o", 150, 250),
    ("gpt-4o-mini", 80, 120),
    ("gpt-4o", 300, 500),
    ("gpt-4o-mini", 120, 180),
]

print("Registrando uso simulado...")
for modelo, t_in, t_out in usos_simulados:
    costo = calculador.registrar(modelo, t_in, t_out)
    print(f"  {modelo}: {t_in}+{t_out} tokens = ${costo:.6f}")

# Resumen actual
resumen = calculador.resumen()
print(f"\n{'='*60}")
print("RESUMEN DE USO ACTUAL")
print(f"{'='*60}")
print(f"Total consultas: {resumen['total_consultas']}")
print(f"Tokens total: {resumen['tokens_total']}")
print(f"Costo total: ${resumen['costo_total_usd']:.6f} USD")
print(f"Presupuesto restante: ${resumen['presupuesto_restante']:.6f} USD")
print(f"Uso del presupuesto: {resumen['porcentaje_presupuesto']}%")
print(f"\nPor modelo:")
for modelo, datos in resumen["por_modelo"].items():
    print(f"  {modelo}: {datos['consultas']} consultas, {datos['tokens']} tokens, ${datos['costo']} USD")

# Proyección de costos
print(f"\n{'='*60}")
print("PROYECCIÓN DE COSTOS (1,000 consultas/día)")
print(f"{'='*60}")
comparacion = calculador.comparar_modelos(consultas_por_dia=1000)
print(f"{'Modelo':<20} {'$/consulta':>12} {'$/día':>10} {'$/mes':>10} {'$/año':>12}")
print("-" * 66)
for c in comparacion:
    print(f"{c['modelo']:<20} ${c['costo_por_consulta']:>10.6f} ${c['costo_diario']:>8.4f} ${c['costo_mensual']:>8.2f} ${c['costo_anual']:>10.2f}")

## 5. Patrones de Resiliencia

Los sistemas en producción deben manejar fallos de forma elegante.
Implementaremos tres patrones clásicos:
- **Retry con backoff exponencial**: Reintentar con espera creciente
- **Circuit breaker**: Desconectar temporalmente un servicio que falla repetidamente
- **Cadena de fallback**: Si el modelo A falla, intentar con el modelo B

In [ ]:
import random


class RetryConBackoff:
    """Implementa reintentos con backoff exponencial y jitter."""

    def __init__(self, max_reintentos: int = 3, base_espera: float = 1.0,
                 factor: float = 2.0, max_espera: float = 30.0):
        self.max_reintentos = max_reintentos
        self.base_espera = base_espera
        self.factor = factor
        self.max_espera = max_espera
        self.historial_reintentos = []

    def ejecutar(self, funcion, *args, **kwargs):
        """Ejecuta una función con reintentos."""
        ultimo_error = None
        intentos = []

        for intento in range(self.max_reintentos + 1):
            try:
                inicio = time.time()
                resultado = funcion(*args, **kwargs)
                latencia = (time.time() - inicio) * 1000

                registro = {
                    "intento": intento + 1,
                    "exito": True,
                    "latencia_ms": round(latencia, 2),
                    "intentos_detalle": intentos
                }
                self.historial_reintentos.append(registro)
                return resultado, registro

            except Exception as e:
                ultimo_error = e
                error_text = str(e)
                es_rate_limit = "RateLimit" in error_text or "429" in error_text
                espera = min(
                    self.base_espera * (self.factor ** intento) + random.uniform(0, 1),
                    self.max_espera
                )
                intentos.append({
                    "intento": intento + 1,
                    "error": error_text,
                    "espera_s": 0 if es_rate_limit else round(espera, 2)
                })

                if es_rate_limit:
                    print("  Rate limit detectado: no se reintenta para evitar esperas largas.")
                    break

                if intento < self.max_reintentos:
                    print(f"  Reintento {intento + 1}/{self.max_reintentos}: "
                          f"esperando {espera:.1f}s... (Error: {error_text[:50]})")
                    time.sleep(espera)

        registro = {
            "intento": self.max_reintentos + 1,
            "exito": False,
            "error": str(ultimo_error),
            "intentos_detalle": intentos
        }
        self.historial_reintentos.append(registro)
        raise ultimo_error


class CircuitBreaker:
    """Implementa el patrón Circuit Breaker para proteger contra fallos en cascada."""

    def __init__(self, umbral_fallos: int = 3, tiempo_recuperacion: float = 30.0):
        """
        umbral_fallos: Número de fallos consecutivos para abrir el circuito.
        tiempo_recuperacion: Segundos antes de intentar cerrar el circuito.
        """
        self.umbral_fallos = umbral_fallos
        self.tiempo_recuperacion = tiempo_recuperacion
        self.fallos_consecutivos = 0
        self.estado = "cerrado"  # cerrado (ok), abierto (bloqueando), semi-abierto (probando)
        self.ultimo_fallo: float | None = None
        self.historial = []

    def puede_ejecutar(self) -> bool:
        """Verifica si el circuito permite ejecutar."""
        if self.estado == "cerrado":
            return True
        elif self.estado == "abierto":
            # Verificar si ya pasó el tiempo de recuperación
            if self.ultimo_fallo and (time.time() - self.ultimo_fallo) >= self.tiempo_recuperacion:
                self.estado = "semi-abierto"
                return True
            return False
        else:  # semi-abierto
            return True

    def registrar_exito(self):
        """Registra una ejecución exitosa."""
        self.fallos_consecutivos = 0
        if self.estado == "semi-abierto":
            self.estado = "cerrado"
        self.historial.append({"timestamp": time.time(), "resultado": "exito", "estado": self.estado})

    def registrar_fallo(self):
        """Registra un fallo."""
        self.fallos_consecutivos += 1
        self.ultimo_fallo = time.time()

        if self.fallos_consecutivos >= self.umbral_fallos:
            self.estado = "abierto"

        self.historial.append({
            "timestamp": time.time(),
            "resultado": "fallo",
            "estado": self.estado,
            "fallos_consecutivos": self.fallos_consecutivos
        })


class CadenaFallback:
    """Implementa una cadena de fallback entre múltiples modelos."""

    def __init__(self, client: OpenAI, modelos: list[str],
                 timeout_por_modelo: float = 10.0):
        self.client = client
        self.modelos = modelos
        self.timeout = timeout_por_modelo
        self.circuit_breakers = {m: CircuitBreaker() for m in modelos}
        self.retry = RetryConBackoff(max_reintentos=2, base_espera=0.5)
        self.historial = []

    def _llamar_modelo(self, modelo: str, pregunta: str) -> str:
        """Llama a un modelo específico."""
        resultado = self.client.chat.completions.create(
            model=modelo,
            messages=[
                {"role": "system", "content": "Responde de forma concisa."},
                {"role": "user", "content": pregunta}
            ],
            temperature=0.5,
            max_tokens=200
        )
        return resultado.choices[0].message.content

    def procesar(self, pregunta: str) -> dict:
        """Procesa una consulta con cadena de fallback."""
        for modelo in self.modelos:
            cb = self.circuit_breakers[modelo]

            if not cb.puede_ejecutar():
                print(f"  Circuito ABIERTO para {modelo}, saltando...")
                continue

            try:
                inicio = time.time()
                respuesta, info_retry = self.retry.ejecutar(
                    self._llamar_modelo, modelo, pregunta
                )
                latencia = (time.time() - inicio) * 1000

                cb.registrar_exito()

                registro = {
                    "modelo_usado": modelo,
                    "respuesta": respuesta,
                    "latencia_ms": round(latencia, 2),
                    "reintentos": info_retry["intento"] - 1,
                    "fallback": modelo != self.modelos[0]
                }
                self.historial.append(registro)
                return registro

            except Exception as e:
                cb.registrar_fallo()
                print(f"  Fallo en {modelo}: {str(e)[:50]}. Intentando fallback...")

        return {
            "modelo_usado": None,
            "respuesta": "Lo sentimos, todos los modelos están temporalmente no disponibles.",
            "error": True
        }


print("Patrones de resiliencia creados correctamente")

In [ ]:
# Probar la cadena de fallback
cadena = CadenaFallback(
    client,
    modelos=[MODELO_POTENTE, MODELO_LIGERO],  # Intentar primero gpt-4o, luego gpt-4o-mini
    timeout_por_modelo=10.0
)

preguntas = [
    "¿Qué es la computación cuántica?",
    "¿Cuáles son los principios SOLID?",
    "¿Qué es un API REST?",
]

print("Procesando con cadena de fallback...\n")
for pregunta in preguntas:
    print(f"Pregunta: {pregunta}")
    resultado = cadena.procesar(pregunta)
    print(f"  Modelo usado: {resultado['modelo_usado']}")
    print(f"  Fallback: {'Sí' if resultado.get('fallback') else 'No'}")
    if 'latencia_ms' in resultado:
        print(f"  Latencia: {resultado['latencia_ms']:.0f}ms")
    print(f"  Respuesta: {resultado['respuesta'][:120]}...\n")

# Estado de los circuit breakers
print(f"\nEstado de Circuit Breakers:")
for modelo, cb in cadena.circuit_breakers.items():
    print(f"  {modelo}: {cb.estado} (fallos: {cb.fallos_consecutivos})")

## 6. Métricas de Sostenibilidad

Recopilamos todas las métricas de las secciones anteriores para generar un reporte
de sostenibilidad con visualizaciones.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


class ReporteSostenibilidad:
    """Genera reportes de sostenibilidad con métricas consolidadas."""

    def __init__(self):
        self.metricas = {
            "tokens_totales": 0,
            "costo_total_usd": 0.0,
            "consultas_totales": 0,
            "cache_hits": 0,
            "cache_misses": 0,
            "tokens_ahorrados_cache": 0,
            "costo_ahorrado_enrutamiento": 0.0,
            "latencias_ms": [],
            "por_modelo": defaultdict(lambda: {"consultas": 0, "tokens": 0, "costo": 0.0})
        }

    def agregar_metricas_cache(self, stats_cache: dict):
        """Agrega métricas del caché."""
        self.metricas["cache_hits"] = stats_cache.get("hits", 0)
        self.metricas["cache_misses"] = stats_cache.get("misses", 0)
        self.metricas["tokens_ahorrados_cache"] = stats_cache.get("tokens_ahorrados_total", 0)

    def agregar_metricas_enrutamiento(self, reporte_enrutamiento: dict):
        """Agrega métricas del enrutamiento."""
        self.metricas["costo_ahorrado_enrutamiento"] = reporte_enrutamiento.get("ahorro_total", 0.0)

    def agregar_registro(self, modelo: str, tokens: int, costo: float, latencia_ms: float):
        """Agrega un registro individual."""
        self.metricas["tokens_totales"] += tokens
        self.metricas["costo_total_usd"] += costo
        self.metricas["consultas_totales"] += 1
        self.metricas["latencias_ms"].append(latencia_ms)
        self.metricas["por_modelo"][modelo]["consultas"] += 1
        self.metricas["por_modelo"][modelo]["tokens"] += tokens
        self.metricas["por_modelo"][modelo]["costo"] += costo

    def generar_reporte(self) -> dict:
        """Genera el reporte completo de sostenibilidad."""
        m = self.metricas
        latencias = m["latencias_ms"]

        total_cache = m["cache_hits"] + m["cache_misses"]
        tasa_cache = (m["cache_hits"] / total_cache * 100) if total_cache > 0 else 0

        reporte = {
            "resumen_general": {
                "consultas_totales": m["consultas_totales"],
                "tokens_totales": m["tokens_totales"],
                "costo_total_usd": round(m["costo_total_usd"], 6),
            },
            "cache": {
                "tasa_hits": round(tasa_cache, 2),
                "tokens_ahorrados": m["tokens_ahorrados_cache"],
                "costo_ahorrado_estimado": round(m["tokens_ahorrados_cache"] / 1000 * 0.00075, 6)
            },
            "enrutamiento": {
                "costo_ahorrado": round(m["costo_ahorrado_enrutamiento"], 6)
            },
            "latencia": {
                "promedio_ms": round(sum(latencias) / len(latencias), 2) if latencias else 0,
                "minima_ms": round(min(latencias), 2) if latencias else 0,
                "maxima_ms": round(max(latencias), 2) if latencias else 0,
            },
            "por_modelo": dict(m["por_modelo"])
        }
        return reporte

    def visualizar(self):
        """Genera gráficos de sostenibilidad."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle("Reporte de Sostenibilidad del Sistema de IA", fontsize=16, fontweight="bold")

        # 1. Distribución de costos por modelo
        ax1 = axes[0, 0]
        modelos = list(self.metricas["por_modelo"].keys())
        costos = [self.metricas["por_modelo"][m]["costo"] for m in modelos]
        if modelos:
            ax1.pie(costos, labels=modelos, autopct="%1.1f%%", startangle=90)
            ax1.set_title("Distribución de Costos por Modelo")
        else:
            ax1.text(0.5, 0.5, "Sin datos", ha="center", va="center")
            ax1.set_title("Distribución de Costos por Modelo")

        # 2. Cache hits vs misses
        ax2 = axes[0, 1]
        cache_data = [self.metricas["cache_hits"], self.metricas["cache_misses"]]
        if sum(cache_data) > 0:
            ax2.bar(["Hits", "Misses"], cache_data, color=["#2ecc71", "#e74c3c"])
            ax2.set_title("Rendimiento del Caché")
            ax2.set_ylabel("Cantidad")
        else:
            ax2.text(0.5, 0.5, "Sin datos de caché", ha="center", va="center")
            ax2.set_title("Rendimiento del Caché")

        # 3. Distribución de latencias
        ax3 = axes[1, 0]
        latencias = self.metricas["latencias_ms"]
        if latencias:
            ax3.hist(latencias, bins=15, color="#3498db", edgecolor="black", alpha=0.7)
            ax3.axvline(sum(latencias)/len(latencias), color="red", linestyle="--", label="Promedio")
            ax3.set_title("Distribución de Latencias")
            ax3.set_xlabel("Latencia (ms)")
            ax3.set_ylabel("Frecuencia")
            ax3.legend()
        else:
            ax3.text(0.5, 0.5, "Sin datos", ha="center", va="center")
            ax3.set_title("Distribución de Latencias")

        # 4. Ahorros
        ax4 = axes[1, 1]
        categorias = ["Caché", "Enrutamiento"]
        ahorros = [
            self.metricas["tokens_ahorrados_cache"] / 1000 * 0.00075,
            self.metricas["costo_ahorrado_enrutamiento"]
        ]
        ax4.barh(categorias, ahorros, color=["#2ecc71", "#9b59b6"])
        ax4.set_title("Ahorros por Estrategia (USD)")
        ax4.set_xlabel("Ahorro (USD)")

        plt.tight_layout()
        plt.show()


print("ReporteSostenibilidad creado correctamente")

In [ ]:
# Generar reporte de sostenibilidad con datos de las secciones anteriores
reporte_sost = ReporteSostenibilidad()

# Agregar métricas del caché
reporte_sost.agregar_metricas_cache(cache.obtener_estadisticas())

# Agregar métricas del enrutamiento
reporte_sost.agregar_metricas_enrutamiento(enrutador.reporte_costos())

# Agregar registros individuales del enrutador
for r in enrutador.historial:
    if "error" not in r:
        reporte_sost.agregar_registro(
            modelo=r["modelo_usado"],
            tokens=r["tokens"]["total"],
            costo=r["costo_real"],
            latencia_ms=r["latencia_ms"]
        )

# Agregar registros de la cadena de fallback
for r in cadena.historial:
    if r.get("modelo_usado"):
        reporte_sost.agregar_registro(
            modelo=r["modelo_usado"],
            tokens=0,  # No tenemos el detalle de tokens aquí
            costo=0.0,
            latencia_ms=r.get("latencia_ms", 0)
        )

# Generar y mostrar reporte
reporte_final = reporte_sost.generar_reporte()
print(json.dumps(reporte_final, indent=2, ensure_ascii=False, default=str))

# Visualizar
reporte_sost.visualizar()

## 7. Ejercicio Final: Sistema Optimizado

Combina **caché + enrutamiento de modelos + resiliencia** para procesar 20 consultas variadas.
Genera un reporte comparando el rendimiento optimizado vs no optimizado.

In [ ]:
class SistemaOptimizado:
    """Sistema que combina caché, enrutamiento y resiliencia."""

    def __init__(self, client: OpenAI):
        self.client = client
        self.cache = CacheSemantico(umbral_similitud=0.75, ttl_segundos=3600)
        self.enrutador = EnrutadorModelos(client)
        self.retry = RetryConBackoff(max_reintentos=2, base_espera=0.5)
        self.calculador = CalculadorCostos(presupuesto_diario=10.0)
        self.resultados = []

    def procesar(self, consulta: str) -> dict:
        """Procesa una consulta con todas las optimizaciones."""
        inicio = time.time()

        # Paso 1: Buscar en caché
        encontrado, respuesta_cache, similitud = self.cache.buscar(consulta)

        if encontrado:
            latencia = (time.time() - inicio) * 1000
            self.cache.registrar_ahorro(tokens=150)
            resultado = {
                "consulta": consulta[:60],
                "fuente": "CACHE",
                "modelo": "N/A",
                "respuesta": respuesta_cache,
                "tokens": 0,
                "costo": 0.0,
                "latencia_ms": round(latencia, 2),
                "similitud_cache": round(similitud, 4)
            }
            self.resultados.append(resultado)
            return resultado

        # Paso 2: Enrutar al modelo apropiado
        complejidad = self.enrutador.clasificar_complejidad(consulta)
        modelo = self.enrutador.seleccionar_modelo(complejidad)

        # Paso 3: Ejecutar con retry
        def llamar_api():
            return self.client.chat.completions.create(
                model=modelo,
                messages=[
                    {"role": "system", "content": "Responde de forma concisa."},
                    {"role": "user", "content": consulta}
                ],
                temperature=0.3,
                max_tokens=200
            )

        try:
            resp_api, info_retry = self.retry.ejecutar(llamar_api)
            respuesta = resp_api.choices[0].message.content
            uso = resp_api.usage

            # Registrar costo
            costos = self.enrutador.COSTOS[modelo]
            costo = (
                (uso.prompt_tokens / 1000) * costos["input"] +
                (uso.completion_tokens / 1000) * costos["output"]
            )
            self.calculador.registrar(modelo, uso.prompt_tokens, uso.completion_tokens)

            # Guardar en caché
            self.cache.guardar(consulta, respuesta, uso.total_tokens)

            latencia = (time.time() - inicio) * 1000
            resultado = {
                "consulta": consulta[:60],
                "fuente": "API",
                "modelo": modelo,
                "complejidad": complejidad,
                "respuesta": respuesta,
                "tokens": uso.total_tokens,
                "costo": round(costo, 6),
                "latencia_ms": round(latencia, 2),
                "reintentos": info_retry["intento"] - 1
            }
            self.resultados.append(resultado)
            return resultado

        except Exception as e:
            latencia = (time.time() - inicio) * 1000
            resultado = {
                "consulta": consulta[:60],
                "fuente": "ERROR",
                "error": str(e),
                "latencia_ms": round(latencia, 2)
            }
            self.resultados.append(resultado)
            return resultado

    def generar_reporte_comparativo(self) -> pd.DataFrame:
        """Genera un reporte comparativo optimizado vs no optimizado."""
        filas = []
        for r in self.resultados:
            fila = {
                "Consulta": r.get("consulta", "")[:40],
                "Fuente": r.get("fuente", "N/A"),
                "Modelo": r.get("modelo", "N/A"),
                "Tokens": r.get("tokens", 0),
                "Costo (USD)": r.get("costo", 0.0),
                "Latencia (ms)": r.get("latencia_ms", 0),
            }
            filas.append(fila)

        df = pd.DataFrame(filas)
        return df


print("SistemaOptimizado creado correctamente")

In [ ]:
# Procesar 20 consultas variadas
sistema = SistemaOptimizado(client)

consultas_20 = [
    # Simples (deberían usar gpt-4o-mini)
    "¿Qué es Python?",
    "¿Qué es JavaScript?",
    "¿Qué es HTML?",
    "¿Qué es CSS?",
    "¿Qué es SQL?",

    # Similares a las anteriores (deberían venir del caché)
    "Qué es el lenguaje Python",
    "Dime qué es JavaScript",
    "Explícame qué es HTML",

    # Medias
    "Explica cómo funciona un API REST",
    "¿Cuál es la diferencia entre TCP y UDP?",
    "¿Por qué es importante la normalización en bases de datos?",
    "Explica el patrón MVC con un ejemplo",

    # Similares (caché)
    "Cómo funciona REST API",
    "Diferencia entre TCP y UDP",

    # Complejas (deberían usar gpt-4o)
    "Analiza las ventajas y desventajas de arquitectura de microservicios",
    "Compara los principales paradigmas de programación y explica en detalle cuándo usar cada uno",
    "Diseña una estrategia de migración de monolito a microservicios con múltiples pasos",

    # Más simples (mix de caché y API)
    "¿Qué es Docker?",
    "¿Qué es Git?",
    "¿Qué es Linux?",
]

print(f"Procesando {len(consultas_20)} consultas con sistema optimizado...\n")
for i, consulta in enumerate(consultas_20):
    resultado = sistema.procesar(consulta)
    icono = "⚡" if resultado["fuente"] == "CACHE" else "🌐"
    costo_str = f"${resultado.get('costo', 0):.6f}" if resultado.get('costo', 0) > 0 else "$0 (cache)"
    print(f"{i+1:2d}. {icono} [{resultado['fuente']:5}] {consulta[:50]:<50} "
          f"| {costo_str:>14} | {resultado.get('latencia_ms', 0):>8.0f}ms")

In [ ]:
# Generar reporte comparativo
df = sistema.generar_reporte_comparativo()
print("\nRESUMEN DEL PROCESAMIENTO")
print("=" * 80)
print(df.to_string(index=False))

# Estadísticas agregadas
total_tokens = df["Tokens"].sum()
total_costo = df["Costo (USD)"].sum()
latencia_prom = df["Latencia (ms)"].mean()
cache_hits = len(df[df["Fuente"] == "CACHE"])
api_calls = len(df[df["Fuente"] == "API"])

# Costo hipotético si todo fuera gpt-4o sin caché
tokens_promedio_por_consulta = total_tokens / max(api_calls, 1)
costo_sin_optimizar = len(consultas_20) * (tokens_promedio_por_consulta / 1000) * 0.0125  # gpt-4o promedio

print(f"\n{'='*60}")
print("COMPARACIÓN: OPTIMIZADO vs NO OPTIMIZADO")
print(f"{'='*60}")
print(f"Total consultas: {len(consultas_20)}")
print(f"Resueltas por caché: {cache_hits} ({cache_hits/len(consultas_20)*100:.1f}%)")
print(f"Llamadas a API: {api_calls} ({api_calls/len(consultas_20)*100:.1f}%)")
print(f"")
print(f"{'Métrica':<30} {'Optimizado':>15} {'Sin optimizar':>15} {'Ahorro':>10}")
print(f"{'-'*70}")
print(f"{'Tokens usados':<30} {total_tokens:>15,} {int(tokens_promedio_por_consulta * 20):>15,}")
print(f"{'Costo (USD)':<30} ${total_costo:>14.6f} ${costo_sin_optimizar:>14.6f} {((costo_sin_optimizar - total_costo) / max(costo_sin_optimizar, 0.000001) * 100):>8.1f}%")
print(f"{'Latencia promedio (ms)':<30} {latencia_prom:>15.0f}")
print(f"{'Tasa de caché (%)':<30} {cache_hits/len(consultas_20)*100:>14.1f}%")

In [ ]:
# Visualización final del rendimiento
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Rendimiento del Sistema Optimizado", fontsize=16, fontweight="bold")

# 1. Fuente de respuestas
ax1 = axes[0, 0]
fuentes = df["Fuente"].value_counts()
ax1.pie(fuentes.values, labels=fuentes.index, autopct="%1.1f%%",
        colors=["#2ecc71", "#3498db", "#e74c3c"])
ax1.set_title("Fuente de Respuestas")

# 2. Latencia por fuente
ax2 = axes[0, 1]
for fuente in df["Fuente"].unique():
    datos = df[df["Fuente"] == fuente]["Latencia (ms)"]
    ax2.bar(fuente, datos.mean(), yerr=datos.std() if len(datos) > 1 else 0,
            capsize=5, alpha=0.8)
ax2.set_title("Latencia Promedio por Fuente")
ax2.set_ylabel("Latencia (ms)")

# 3. Costo acumulado
ax3 = axes[1, 0]
costos_acumulados = df["Costo (USD)"].cumsum()
ax3.plot(range(1, len(costos_acumulados) + 1), costos_acumulados, marker="o",
         markersize=4, linewidth=2, color="#e74c3c")
ax3.set_title("Costo Acumulado por Consulta")
ax3.set_xlabel("Número de Consulta")
ax3.set_ylabel("Costo Acumulado (USD)")
ax3.grid(True, alpha=0.3)

# 4. Tokens por consulta
ax4 = axes[1, 1]
colores = ["#2ecc71" if f == "CACHE" else "#3498db" for f in df["Fuente"]]
ax4.bar(range(1, len(df) + 1), df["Tokens"], color=colores, alpha=0.8)
ax4.set_title("Tokens por Consulta (verde=caché, azul=API)")
ax4.set_xlabel("Número de Consulta")
ax4.set_ylabel("Tokens")

plt.tight_layout()
plt.show()

print("\nReporte generado exitosamente.")

## Conclusiones

En este notebook implementamos las siguientes estrategias de escalabilidad y sostenibilidad:

1. **Caché Semántico**: Reutiliza respuestas para consultas similares, reduciendo llamadas a la API y latencia.
2. **Procesamiento por Lotes**: Agrupa solicitudes con cola de prioridad para procesamiento eficiente.
3. **Enrutamiento de Modelos**: Dirige consultas simples a modelos baratos y consultas complejas a modelos potentes.
4. **Estimación de Costos**: Proyecta gastos y genera alertas de presupuesto.
5. **Patrones de Resiliencia**: Retry con backoff, circuit breaker y cadena de fallback.
6. **Métricas de Sostenibilidad**: Monitoreo consolidado con visualizaciones.

### Impacto Medido
- **Caché**: Reduce llamadas a la API y latencia a milisegundos para consultas repetidas.
- **Enrutamiento**: Ahorra costos al usar modelos apropiados para cada complejidad.
- **Resiliencia**: Garantiza disponibilidad del servicio ante fallos de proveedores.

### Recomendaciones para Producción
- Usar un caché distribuido (Redis) en lugar de memoria local.
- Implementar colas de mensajes (RabbitMQ, Kafka) para procesamiento asíncrono.
- Monitorear métricas con herramientas como Prometheus + Grafana.
- Establecer presupuestos por equipo/proyecto y alertas automatizadas.
- Revisar periódicamente la efectividad del caché y ajustar umbrales.